# **VI.  Análisis de variables numéricas**

## Objetivo
Este notebook tiene la función de analizar la distribución, propiedades estadísticas y anomalías de variables numéricas presentes en el dataset oncológico de GRD (2019-2024), permitiendo identificar valores atípicos, nulos, ceros y patrones temporales. El análisis genera visualizaciones (boxplots) y reportes de terciles para evaluar la calidad de datos numéricos.

## Proceso
1. **Configuración inicial**: Especificar rutas de datos y variables numéricas a analizar
2. **Limpieza de datos numéricos**: Conversión de strings a números, manejo de decimales con coma, valores problemáticos
3. **Cálculo de estadísticas**: Media, mediana, desviación estándar, cuantiles, coeficiente de variación
4. **Detección de outliers**: Identificación usando regla IQR (Interquartile Range)
5. **Análisis de distribución**: Clasificación en terciles (BAJO, MEDIO, ALTO)
6. **Generación de visualizaciones**: Boxplots por año para cada variable
7. **Exportación de resultados**: Tablas resumen, gráficos y distribuciones en CSV/PNG

## Descripciones de columnas de salida:

| Columna | Significado |
|---------|-------------|
| **Año** | Año del análisis (2019-2024) |
| **Variable** | Nombre de la variable numérica analizada |
| **Total** | Número total de registros |
| **Nulos** | Cantidad de valores faltantes (NaN, vacíos) |
| **Ceros** | Cantidad de valores exactamente 0 |
| **Min/Max** | Valores mínimo y máximo |
| **Media** | Promedio aritmético |
| **Mediana** | Percentil 50% (valor central) |
| **Std** | Desviación estándar |
| **CV** | Coeficiente de Variación (Std/Media): mide variabilidad relativa |
| **Q1/Q2/Q3** | Cuartiles (25%, 50%, 75%) |
| **IQR** | Rango Intercuartílico (Q3-Q1) |
| **Outliers** | Cantidad de valores atípicos según regla IQR |
| **% Nulos** | Porcentaje de datos faltantes |
| **% Outliers** | Porcentaje de outliers respecto al total |

In [1]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================

import pandas as pd 
import numpy as np 
import os 
import matplotlib.pyplot as plt 
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

# Ruta de salida para resultados numéricos segmentados
ruta_resultados_num = "../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas"
os.makedirs(ruta_resultados_num, exist_ok=True)

# Ruta de datos CLASIFICADOS (Masivos)
carpeta = "../../Datos/Datos clasificados"

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"
}

In [2]:
# ============================================================================
# UTILIDADES: Limpieza Numérica
# ============================================================================
def limpiar_columna_numerica(col):
    col = col.astype(str).str.strip()
    col = col.str.replace(",", ".", regex=False)
    col = col.str.replace(r"^\.", "0.", regex=True)
    col = col.str.replace(r"^,", "0.", regex=True)
    return pd.to_numeric(col, errors="coerce")


# ============================================================================
# ANÁLISIS DE VARIABLES NUMÉRICAS SEGMENTADAS
# ============================================================================

def analizar_variable_numerica(carpeta, archivos, mapa_columnas, columna):
    """
    Lee los archivos masivos de forma optimizada, limpia la variable numérica, 
    y genera estadísticas, outliers y terciles separados para la cohorte Total, 
    Oncológica y Control de manera simultánea.
    """
    print(f"\n[{columna}] Extrayendo y calculando estadísticas segmentadas...")
    
    # Listas para guardar los diccionarios de resultados
    res_total, res_onco, res_control = [], [], []

    # Crear carpetas de salida
    carpeta_var = os.path.join(ruta_resultados_num, columna)
    for sub in ["Total", "Oncológicos", "Control"]:
        os.makedirs(os.path.join(carpeta_var, sub), exist_ok=True)

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año = int(archivo[-8:-4])

        # 1. OPTIMIZACIÓN: Leer solo columna y CATEGORIA_CANCER
        encabezados = pd.read_csv(ruta, nrows=0).rename(columns=mapa_columnas).columns
        if columna not in encabezados:
            continue
            
        columnas_a_leer = [col for col in pd.read_csv(ruta, nrows=0).columns 
                           if col == columna or col == 'CATEGORIA_CANCER' or mapa_columnas.get(col) == columna]
        
        df = pd.read_csv(ruta, usecols=columnas_a_leer, low_memory=False).rename(columns=mapa_columnas)
        
        if df.empty: continue

        # 2. Limpieza global en RAM
        df[columna] = limpiar_columna_numerica(df[columna])

        # 3. Separación de Poblaciones
        mask_control = df['CATEGORIA_CANCER'] == 'SIN_CANCER'
        poblaciones = {
            "Total": df,
            "Oncológicos": df[~mask_control],
            "Control": df[mask_control]
        }

        # 4. Cálculo iterativo para cada población
        for nombre_pob, df_pob in poblaciones.items():
            col_pob = df_pob[columna]
            total = len(col_pob)
            n_null = col_pob.isna().sum()
            n_ceros = (col_pob == 0).sum()
            col_clean = col_pob.dropna()

            if len(col_clean) == 0:
                continue

            # Estadísticas
            minimo, maximo = col_clean.min(), col_clean.max()
            media, std = col_clean.mean(), col_clean.std()
            q1, mediana, q3 = col_clean.quantile([0.25, 0.50, 0.75])
            iqr = q3 - q1
            cv = std / media if media != 0 else None

            # Outliers (IQR)
            outliers = col_clean[(col_clean < (q1 - 1.5 * iqr)) | (col_clean > (q3 + 1.5 * iqr))]
            n_outliers = len(outliers)

            # Terciles
            terciles = col_clean.quantile([0.33, 0.66]).values
            if len(set(terciles)) < 2:
                dist_terciles = pd.Series(["UNICO"] * len(col_clean)).value_counts()
            else:
                categorias = pd.cut(col_clean, bins=[-np.inf, terciles[0], terciles[1], np.inf], labels=["BAJO", "MEDIO", "ALTO"], duplicates="drop")
                dist_terciles = categorias.value_counts()

            # Guardar métricas en la lista correspondiente
            fila = {
                "Año": año, "Variable": columna, "Total": total, "Nulos": n_null, "Ceros": n_ceros,
                "Min": minimo, "Max": maximo, "Media": round(media, 2), "Mediana": round(mediana, 2),
                "Std": round(std, 2), "CV": round(cv, 3) if cv else None, "Q1": round(q1, 2), "Q2": round(mediana, 2), "Q3": round(q3, 2),
                "IQR": round(iqr, 2), "Outliers": n_outliers, "% Nulos": round((n_null/total)*100, 2), "% Outliers": round((n_outliers/total)*100, 2)
            }

            if nombre_pob == "Total": res_total.append(fila)
            elif nombre_pob == "Oncológicos": res_onco.append(fila)
            else: res_control.append(fila)

            # Guardar Terciles y Boxplots
            sub_ruta = os.path.join(carpeta_var, nombre_pob)
            dist_terciles.to_csv(os.path.join(sub_ruta, f"{columna.lower()}_{año}_terciles.csv"), encoding="utf-8-sig")

            plt.figure(figsize=(6, 4))
            plt.boxplot(col_clean)
            plt.title(f"Boxplot: {columna} ({año}) - {nombre_pob}")
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(sub_ruta, f"{columna.lower()}_{año}_boxplot.png"), dpi=100, bbox_inches='tight')
            plt.close()

    # =====================================================================
    # TABLAS FINALES
    # =====================================================================
    def guardar_resumen(resultados, nombre):
        if resultados:
            df_res = pd.DataFrame(resultados).sort_values("Año")
            df_res.to_csv(os.path.join(carpeta_var, nombre, f"{columna.lower()}_resumen_{nombre}.csv"), index=False, encoding="utf-8-sig")
            return df_res
        return None

    df_tot = guardar_resumen(res_total, "Total")
    df_onc = guardar_resumen(res_onco, "Oncológicos")
    df_ctrl = guardar_resumen(res_control, "Control")

    print(f"✓ Análisis guardado en 3 subcarpetas dentro de: {carpeta_var}")
    return df_tot, df_onc, df_ctrl

def mostrar_resultados(matriz_total, matriz_onco, matriz_control, variable):
    mensaje_total = f"1. Población TOTAL (análisis de {variable}):"
    mensaje_oncologico = f"2. Población ONCOLÓGICA (análisis de {variable}):"
    mensaje_control = f"3. Población CONTROL (análisis de {variable}):"
    display(mensaje_total)
    display(matriz_total)
    display(mensaje_oncologico)
    display(matriz_onco)
    display(mensaje_control)
    display(matriz_control)

## **1. Variable numérica objetivo:**

### **1. Consumo de Recursos / Peso Relativo GRD (IR_29301_PESO)**
- **Tipo de variable**: Clínica / Financiera (Variable Objetivo - *Target*).
- **Descripción**: Expresión numérica continua que refleja la intensidad del consumo de recursos de un episodio hospitalario en relación con el caso promedio nacional (donde 1.0 es el estándar).
- **Análisis Estadístico**: 
  - **Calidad excelente:** Presenta 0% de nulos en casi todos los años (máximo 2 nulos sobre miles de registros).
  - **Contraste Epidemiológico:** La ejecución segmentada demostró una brecha masiva en el consumo. La mediana del grupo oncológico se sitúa sostenidamente por encima de 1.03, mientras que el grupo control oscila en 0.66. Esto prueba matemáticamente que el paciente oncológico base requiere un 50% más de esfuerzo clínico/financiero que el paciente estándar.
  - **Asimetría y Cola Pesada:** Se detectó entre un 7% y un 9% de valores atípicos (*outliers*) superiores en el grupo oncológico, con máximos que alcanzan los 17.14 puntos de peso. Estos *outliers* no son errores, sino que representan a la subpoblación crítica (estadías prolongadas, múltiples cirugías, uso intensivo de UCI).
- **Veredicto**: **Discretizar en Terciles**. Al ser la variable objetivo para perfilar el consumo, se mantendrá en su forma continua para análisis descriptivos, pero se transformará en una variable categórica ordinal (`CONSUMO_BAJO`, `CONSUMO_MEDIO`, `CONSUMO_ALTO`) mediante *q-cut* (cuantiles). Esta discretización permitirá aplicar algoritmos de clasificación y perfilamiento, garantizando el balanceo de clases necesario para que los selectores de características funcionen correctamente.

In [4]:
# ============================================================================
# 1. Análisis de PESO (Variable IR_29301_PESO)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: IR_29301_PESO (Consumo de recursos del paciente)")
print("="*100 + "\n")

df_total, df_onco, df_control = analizar_variable_numerica(carpeta, archivos, mapa_columnas, "IR_29301_PESO")
mostrar_resultados(df_total, df_onco, df_control, "IR_29301_PESO")


ANALIZANDO VARIABLE: IR_29301_PESO (Consumo de recursos del paciente)


[IR_29301_PESO] Extrayendo y calculando estadísticas segmentadas...
✓ Análisis guardado en 3 subcarpetas dentro de: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\IR_29301_PESO


'1. Población TOTAL (análisis de IR_29301_PESO):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,IR_29301_PESO,1151398,21,470,0.0,20.6461,0.84,0.63,0.92,1.090,0.44,0.63,0.96,0.52,69619,0.0,6.05
1,2020,IR_29301_PESO,781911,0,190,0.0,20.6461,1.01,0.69,1.21,1.206,0.47,0.69,1.09,0.61,56307,0.0,7.20
2,2021,IR_29301_PESO,816909,0,365,0.0,20.6461,1.11,0.71,1.41,1.272,0.49,0.71,1.18,0.69,57798,0.0,7.08
3,2022,IR_29301_PESO,932837,23,724,0.0,20.6461,0.97,0.69,1.14,1.172,0.47,0.69,1.04,0.57,70520,0.0,7.56
4,2023,IR_29301_PESO,1039587,30,1682,0.0,20.6461,0.96,0.69,1.09,1.136,0.47,0.69,1.03,0.56,78487,0.0,7.55
5,2024,IR_29301_PESO,1085813,15,1475,0.0,20.6461,0.97,0.69,1.09,1.125,0.48,0.69,1.04,0.56,82593,0.0,7.61


'2. Población ONCOLÓGICA (análisis de IR_29301_PESO):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,IR_29301_PESO,97829,0,13,0.0,17.1414,1.16,0.97,0.95,0.820,0.74,0.97,1.27,0.53,7063,0.0,7.22
1,2020,IR_29301_PESO,64926,0,4,0.0,14.2492,1.32,1.02,1.14,0.863,0.77,1.02,1.40,0.63,5349,0.0,8.24
2,2021,IR_29301_PESO,67926,0,21,0.0,17.1414,1.35,1.05,1.23,0.905,0.77,1.05,1.46,0.70,5065,0.0,7.46
3,2022,IR_29301_PESO,78069,2,47,0.0,14.2492,1.34,1.04,1.22,0.914,0.76,1.04,1.40,0.64,6992,0.0,8.96
4,2023,IR_29301_PESO,90715,0,80,0.0,14.2492,1.33,1.03,1.19,0.894,0.76,1.03,1.39,0.63,8437,0.0,9.30
5,2024,IR_29301_PESO,99153,2,76,0.0,14.2492,1.33,1.03,1.20,0.900,0.74,1.03,1.39,0.65,8882,0.0,8.96


'3. Población CONTROL (análisis de IR_29301_PESO):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,IR_29301_PESO,1053569,21,457,0.0,20.6461,0.82,0.59,0.91,1.119,0.44,0.59,0.88,0.45,81803,0.0,7.76
1,2020,IR_29301_PESO,716985,0,186,0.0,20.6461,0.98,0.67,1.22,1.243,0.46,0.67,1.04,0.58,51466,0.0,7.18
2,2021,IR_29301_PESO,748983,0,344,0.0,20.6461,1.08,0.69,1.42,1.310,0.47,0.69,1.13,0.65,60849,0.0,8.12
3,2022,IR_29301_PESO,854768,21,677,0.0,20.6461,0.94,0.66,1.13,1.199,0.44,0.66,1.01,0.57,59610,0.0,6.97
4,2023,IR_29301_PESO,948872,30,1602,0.0,20.6461,0.93,0.66,1.07,1.162,0.44,0.66,1.01,0.57,64099,0.0,6.76
5,2024,IR_29301_PESO,986660,13,1399,0.0,20.6461,0.93,0.67,1.07,1.149,0.46,0.67,1.03,0.56,66785,0.0,6.77


## **2. Resto de variables numéricas:**

### **2. Pesos de Recién Nacidos (PESORN1 al PESORN4)**
- **Tipo de variable**: Hospitalaria (Resultados Neonatales).
- **Descripción**: Registro numérico (en gramos) del peso al nacer de hasta cuatro bebés producto de un episodio de hospitalización con parto.
- **Análisis Estadístico**: Densidad prácticamente nula en la población de interés. En la cohorte oncológica, la variable `PESORN1` alcanza un 99.9% de valores nulos, mientras que `PESORN2`, `PESORN3` y `PESORN4` están un 100% vacías en casi todos los períodos observados.
- **Veredicto**: **Descartar completamente**. Al igual que se concluyó con las variables categóricas neonatales, la cohorte de este estudio está delimitada a episodios cuyo diagnóstico principal o complicación basal es una neoplasia maligna (C00-C97). Que una paciente oncológica en tratamiento activo curse un parto durante la misma hospitalización es un evento estadísticamente marginal (*outlier* clínico a nivel poblacional). Retener estas variables inyectaría cuatro columnas de ruido absoluto (llenas de valores nulos y ceros) al conjunto de datos, consumiendo memoria sin aportar varianza matemática que los algoritmos puedan utilizar para el perfilamiento.

In [5]:
# ============================================================================
# 2. Análisis de PESORN1 (Consumo de recursos del Primer Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN1 (Consumo de recursos del primer recién nacido)")
print("="*100 + "\n")
df_total, df_onco, df_control = analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN1")
mostrar_resultados(df_total, df_onco, df_control, "PESORN1")


ANALIZANDO VARIABLE: PESORN1 (Consumo de recursos del primer recién nacido)


[PESORN1] Extrayendo y calculando estadísticas segmentadas...
✓ Análisis guardado en 3 subcarpetas dentro de: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN1


'1. Población TOTAL (análisis de PESORN1):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN1,1151398,994762,24,0.0,9070.0,3204.41,3285.0,672.72,0.210,2905.0,3285.0,3620.0,715.0,7260,86.40,0.63
1,2020,PESORN1,781911,647983,36,0.0,7340.0,3201.48,3285.0,685.26,0.214,2890.0,3285.0,3630.0,740.0,6121,82.87,0.78
2,2021,PESORN1,816909,697825,0,105.0,9834.0,3179.52,3270.0,694.93,0.219,2860.0,3270.0,3620.0,760.0,5262,85.42,0.64
3,2022,PESORN1,932837,797341,0,100.0,7085.0,3176.54,3260.0,675.62,0.213,2865.0,3260.0,3600.0,735.0,5931,85.47,0.64
4,2023,PESORN1,1039587,908370,0,110.0,8900.0,3166.00,3250.0,673.53,0.213,2855.0,3250.0,3594.0,739.0,5613,87.38,0.54
5,2024,PESORN1,1085813,966203,0,100.0,9730.0,3166.21,3250.0,668.38,0.211,2860.0,3250.0,3590.0,730.0,5164,88.98,0.48


'2. Población ONCOLÓGICA (análisis de PESORN1):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN1,97829,97748,0,500.0,4816.0,3083.94,3258.0,714.17,0.232,2630.00,3258.0,3540.00,910.0,2,99.92,0.00
1,2020,PESORN1,64926,64843,2,0.0,4700.0,2968.95,3210.0,894.21,0.301,2552.50,3210.0,3595.00,1042.5,3,99.87,0.00
2,2021,PESORN1,67926,67832,0,480.0,4350.0,2916.62,3071.5,851.80,0.292,2511.25,3071.5,3478.75,967.5,6,99.86,0.01
3,2022,PESORN1,78069,77940,0,400.0,4600.0,2992.35,3110.0,809.22,0.270,2470.00,3110.0,3560.00,1090.0,2,99.83,0.00
4,2023,PESORN1,90715,90595,0,420.0,4380.0,2962.29,3167.5,813.21,0.275,2520.50,3167.5,3492.50,972.0,4,99.87,0.00
5,2024,PESORN1,99153,99024,0,100.0,4338.0,2964.71,3160.0,833.93,0.281,2630.00,3160.0,3510.00,880.0,8,99.87,0.01


'3. Población CONTROL (análisis de PESORN1):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN1,1053569,897014,24,0.0,9070.0,3204.47,3285.0,672.69,0.210,2905.0,3285.0,3620.0,715.0,7255,85.14,0.69
1,2020,PESORN1,716985,583140,34,0.0,7340.0,3201.63,3285.0,685.09,0.214,2890.0,3285.0,3630.0,740.0,6113,81.33,0.85
2,2021,PESORN1,748983,629993,0,105.0,9834.0,3179.73,3270.0,694.75,0.218,2860.0,3270.0,3620.0,760.0,5253,84.11,0.70
3,2022,PESORN1,854768,719401,0,100.0,7085.0,3176.72,3260.0,675.46,0.213,2865.0,3260.0,3600.0,735.0,5920,84.16,0.69
4,2023,PESORN1,948872,817775,0,110.0,8900.0,3166.19,3250.0,673.37,0.213,2855.0,3250.0,3595.0,740.0,5583,86.18,0.59
5,2024,PESORN1,986660,867179,0,100.0,9730.0,3166.43,3250.0,668.15,0.211,2860.0,3250.0,3590.0,730.0,5153,87.89,0.52


In [6]:
# ============================================================================
# 3. Análisis de PESORN2 (Consumo de recursos delSegundo Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN2 (Consumo de recursos del segundo recién nacido)")
print("="*100 + "\n")
df_total, df_onco, df_control = analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN2")
mostrar_resultados(df_total, df_onco, df_control, "PESORN2")


ANALIZANDO VARIABLE: PESORN2 (Consumo de recursos del segundo recién nacido)


[PESORN2] Extrayendo y calculando estadísticas segmentadas...
✓ Análisis guardado en 3 subcarpetas dentro de: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN2


'1. Población TOTAL (análisis de PESORN2):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN2,1151398,1149822,4,0.0,3980.0,2266.63,2360.0,665.07,0.293,1960.0,2360.0,2700.00,740.00,80,99.86,0.01
1,2020,PESORN2,781911,780491,2,0.0,4300.0,2250.39,2350.0,677.66,0.301,1890.0,2350.0,2705.00,815.00,44,99.82,0.01
2,2021,PESORN2,816909,815547,0,120.0,3790.0,2282.70,2345.0,615.49,0.270,1956.0,2345.0,2708.75,752.75,38,99.83,0.00
3,2022,PESORN2,932837,931312,0,100.0,4040.0,2256.87,2346.0,635.76,0.282,1940.0,2346.0,2680.00,740.00,61,99.84,0.01
4,2023,PESORN2,1039587,1038192,0,110.0,4420.0,2256.03,2342.0,623.01,0.276,1925.0,2342.0,2675.00,750.00,45,99.87,0.00
5,2024,PESORN2,1085813,1084495,0,150.0,3960.0,2245.73,2332.5,628.92,0.280,1905.0,2332.5,2690.00,785.00,35,99.88,0.00


'2. Población ONCOLÓGICA (análisis de PESORN2):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2020,PESORN2,64926,64923,0,1120.0,3110.0,2185.00,2325.0,1002.36,0.459,1722.50,2325.0,2717.5,995.00,0,100.0,0.0
1,2021,PESORN2,67926,67924,0,1976.0,2730.0,2353.00,2353.0,533.16,0.227,2164.50,2353.0,2541.5,377.00,0,100.0,0.0
2,2023,PESORN2,90715,90711,0,1185.0,3100.0,2081.25,2020.0,784.99,0.377,1811.25,2020.0,2290.0,478.75,1,100.0,0.0


'3. Población CONTROL (análisis de PESORN2):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN2,1053569,1051993,4,0.0,3980.0,2266.63,2360.0,665.07,0.293,1960.0,2360.0,2700.00,740.00,80,99.85,0.01
1,2020,PESORN2,716985,715568,2,0.0,4300.0,2250.53,2350.0,677.32,0.301,1890.0,2350.0,2705.00,815.00,44,99.80,0.01
2,2021,PESORN2,748983,747623,0,120.0,3790.0,2282.60,2345.0,615.76,0.270,1955.0,2345.0,2706.25,751.25,38,99.82,0.01
3,2022,PESORN2,854768,853243,0,100.0,4040.0,2256.87,2346.0,635.76,0.282,1940.0,2346.0,2680.00,740.00,61,99.82,0.01
4,2023,PESORN2,948872,947481,0,110.0,4420.0,2256.53,2345.0,622.77,0.276,1925.0,2345.0,2675.00,750.00,45,99.85,0.00
5,2024,PESORN2,986660,985342,0,150.0,3960.0,2245.73,2332.5,628.92,0.280,1905.0,2332.5,2690.00,785.00,35,99.87,0.00


In [7]:
# ============================================================================
# 4. Análisis de PESORN3 (Consumo de recursos del Tercer Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN3 (Consumo de recursos del tercer recién nacido)")
print("="*100 + "\n")
df_total, df_onco, df_control = analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN3")
mostrar_resultados(df_total, df_onco, df_control, "PESORN3")


ANALIZANDO VARIABLE: PESORN3 (Consumo de recursos del tercer recién nacido)


[PESORN3] Extrayendo y calculando estadísticas segmentadas...
✓ Análisis guardado en 3 subcarpetas dentro de: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN3


'1. Población TOTAL (análisis de PESORN3):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN3,1151398,1151385,0,555.0,1970.0,1252.46,1294.0,458.76,0.366,800.0,1294.0,1556.0,756.0,0,100.0,0.0
1,2020,PESORN3,781911,781897,0,265.0,2230.0,1594.07,1690.0,526.99,0.331,1437.5,1690.0,2004.0,566.5,1,100.0,0.0
2,2021,PESORN3,816909,816900,0,553.0,2075.0,1455.00,1520.0,505.12,0.347,1073.0,1520.0,1820.0,747.0,0,100.0,0.0
3,2022,PESORN3,932837,932826,0,820.0,3400.0,2033.73,1920.0,782.69,0.385,1620.0,1920.0,2216.0,596.0,2,100.0,0.0
4,2023,PESORN3,1039587,1039574,0,260.0,2100.0,1300.85,1490.0,615.98,0.474,1005.0,1490.0,1791.0,786.0,0,100.0,0.0
5,2024,PESORN3,1085813,1085798,0,100.0,2315.0,1667.53,1755.0,590.40,0.354,1557.0,1755.0,2050.0,493.0,2,100.0,0.0


'2. Población ONCOLÓGICA (análisis de PESORN3):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2023,PESORN3,90715,90714,0,1240.0,1240.0,1240.0,1240.0,NaN,NaN,1240.0,1240.0,1240.0,0.0,0,100.0,0.0


'3. Población CONTROL (análisis de PESORN3):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN3,1053569,1053556,0,555.0,1970.0,1252.46,1294.0,458.76,0.366,800.0,1294.0,1556.00,756.00,0,100.0,0.0
1,2020,PESORN3,716985,716971,0,265.0,2230.0,1594.07,1690.0,526.99,0.331,1437.5,1690.0,2004.00,566.50,1,100.0,0.0
2,2021,PESORN3,748983,748974,0,553.0,2075.0,1455.00,1520.0,505.12,0.347,1073.0,1520.0,1820.00,747.00,0,100.0,0.0
3,2022,PESORN3,854768,854757,0,820.0,3400.0,2033.73,1920.0,782.69,0.385,1620.0,1920.0,2216.00,596.00,2,100.0,0.0
4,2023,PESORN3,948872,948860,0,260.0,2100.0,1305.92,1515.0,643.08,0.492,934.5,1515.0,1798.25,863.75,0,100.0,0.0
5,2024,PESORN3,986660,986645,0,100.0,2315.0,1667.53,1755.0,590.40,0.354,1557.0,1755.0,2050.00,493.00,2,100.0,0.0


In [8]:
# ============================================================================
# 5. Análisis de PESORN4 (Consumo de recursos del cuarto recién nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN4 (Consumo de recursos del cuarto recién nacido)")
print("="*100 + "\n")
df_total, df_onco, df_control = analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN4")
mostrar_resultados(df_total, df_onco, df_control, "PESORN4")


ANALIZANDO VARIABLE: PESORN4 (Consumo de recursos del cuarto recién nacido)


[PESORN4] Extrayendo y calculando estadísticas segmentadas...
✓ Análisis guardado en 3 subcarpetas dentro de: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN4


'1. Población TOTAL (análisis de PESORN4):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2020,PESORN4,781911,781910,0,780.0,780.0,780.0,780.0,NaN,NaN,780.0,780.0,780.0,0.0,0,100.0,0.0


'2. Población ONCOLÓGICA (análisis de PESORN4):'

None

'3. Población CONTROL (análisis de PESORN4):'

,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2020,PESORN4,716985,716984,0,780.0,780.0,780.0,780.0,NaN,NaN,780.0,780.0,780.0,0.0,0,100.0,0.0
